In [ ]:
import matplotlib.pyplot as plt
import os
import pandas as pd
import numpy as np
from PIL import Image

In [ ]:
image_paths = './images/' + pd.Series(os.listdir('./images'))

In [ ]:
df = pd.DataFrame(list(image_paths),columns=['image_path'])
df['image'] = df.image_path.map(Image.open)

In [ ]:
def split_name(image_path):
    name = ''.join(image_path.split('/')[-1])[0:-4].split('-')
    return pd.Series([
        int(name[0]),
        int(name[1]),
        int(name[2]),
    ])
df[['gid', 'sgid', 'fileid']] = df.image_path.apply(split_name)

In [ ]:
def get_resolution(img):
    w,h = img.size
    return pd.Series([w,h])
df[['width', 'height']] = df.image.apply(get_resolution)
df['resolution'] = df.width.astype(str) + 'x' + df.height.astype(str)
print('Unique resolutions:', df.resolution.unique())
df.resolution.value_counts().head(5)

In [ ]:
# Filter

df = df[df.resolution.eq('4100x3148')]
df = df.reset_index(drop=True)

In [ ]:
page = df.loc[86]

In [ ]:
img = page.image.crop((0,0,page.image.width//2, page.image.height))
hcut = 1- (np.array(img)/255).mean(axis=2).mean(axis=1)
hcut_mean = np.mean(hcut)
hcut_idxs = (hcut<hcut_mean).nonzero()[0]
hcut1,hcut2 = hcut_idxs[0], hcut_idxs[-1]

vcut = 1- (np.array(img)/255).mean(axis=0).mean(axis=1)
vcut_mean = np.mean(vcut)
vcut_idxs = (vcut<vcut_mean).nonzero()[0]
vcut1,vcut2 = vcut_idxs[0], vcut_idxs[-1]
plt.plot(vcut[vcut1:vcut2])

hcut1,vcut1,vcut2,hcut2 = hcut1+50,vcut1+5,vcut2,hcut2-25 # Margins
vcut1 += 53 # Remove heading

img = page.image.crop((hcut1,vcut1,vcut2, hcut2))
img

In [ ]:
import requests
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM 
import torch

# 1. Konfiguracja urządzenia i ładowanie modelu
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True).to(device)
processor = AutoProcessor.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True)

# 2. Przygotowanie obrazu i promptu
# Możesz podmienić URL na lokalną ścieżkę: Image.open("moje_zdjecie.jpg")
image = img.resize((1024,1024))

# Prompt: <CAPTION_TO_PHRASE_GROUNDING> służy do znajdowania fraz na obrazie
# prompt = "<CAPTION_TO_PHRASE_GROUNDING> "
prompt = '<CAPTION_TO_PHRASE_GROUNDING> Word Nomen'

# 3. Przetwarzanie (Inference)
inputs = processor(text=prompt, images=image, return_tensors="pt").to(device)

generated_ids = model.generate(
    input_ids=inputs["input_ids"],
    pixel_values=inputs["pixel_values"],
    max_new_tokens=1024,
    num_beams=3
)

generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]

# 4. Post-processing (wyciąganie współrzędnych)
results = processor.post_process_generation(
    generated_text, 
    task="<CAPTION_TO_PHRASE_GROUNDING>", 
    image_size=(image.width, image.height)
)
print(results)
print(f"Wynik detekcji: {results['<CAPTION_TO_PHRASE_GROUNDING>']}")

data = results["<CAPTION_TO_PHRASE_GROUNDING>"]

# 6. Wyświetlanie obrazu i rysowanie kropek
plt.figure(figsize=(12, 8))
plt.imshow(image)
ax = plt.gca()

# Iterujemy po znalezionych ramkach (bboxes) i etykietach
for bbox, label in zip(data['bboxes'], data['labels']):
    # Wyliczamy środek ramki (punkt)
    x_center = (bbox[0] + bbox[2]) / 2
    y_center = (bbox[1] + bbox[3]) / 2
    
    # Rysujemy kropkę (czerwona kropka 'ro')
    ax.plot(x_center, y_center, 'ro', markersize=10, markeredgecolor='white', label=label)
    
    # Dodajemy tekst obok kropki
    plt.text(x_center + 150, y_center - 10, label, color='white', 
             fontsize=12, fontweight='bold', backgroundcolor='red')

plt.axis('off') # Ukrywamy osie
plt.title(f"Detekcja punktowa dla: {prompt}")
plt.show()

In [ ]:
import torch, pytesseract
from transformers import TableTransformerForObjectDetection, DetrImageProcessor
from PIL import Image

# 1. Load image and models
processor = DetrImageProcessor.from_pretrained("microsoft/table-transformer-structure-recognition")
model = TableTransformerForObjectDetection.from_pretrained("microsoft/table-transformer-structure-recognition")

# 2. Process image
image = img
inputs = processor(images=image, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

# 3. Get bounding boxes
results = processor.post_process_object_detection(outputs, threshold=0.6, target_sizes=[image.size[::-1]])[0]
print(results)
rows, cols = [], []

for label, box in zip(results['labels'], results['boxes']):
    label_name = model.config.id2label[label.item()]
    if label_name == 'table row': rows.append(box.tolist())
    if label_name == 'table column': cols.append(box.tolist())

# 4. Sort (Rows top-to-bottom, Cols left-to-right)
rows.sort(key=lambda x: x[1])
cols.sort(key=lambda x: x[0])

# 5. Get 2nd row (index 1) and 3rd column (index 2)
target_row, target_col = rows[1], cols[2]

# 6. Calculate intersection, crop, and read text
cell_box = (
    max(target_row[0], target_col[0]), max(target_row[1], target_col[1]),
    min(target_row[2], target_col[2]), min(target_row[3], target_col[3])
)

# text = pytesseract.image_to_string(image.crop(cell_box)).strip()
# print(text)
image.crop(cell_box)

In [ ]:
target_row, target_col = rows[6], cols[2]

# 6. Calculate intersection, crop, and read text
cell_box = (
    max(target_row[0], target_col[0]), max(target_row[1], target_col[1]),
    min(target_row[2], target_col[2]), min(target_row[3], target_col[3])
)

# text = pytesseract.image_to_string(image.crop(cell_box)).strip()
# print(text)
image.crop(cell_box)

In [ ]:
import cv2
import numpy as np
from PIL import Image

def get_straightening_rotation_matrix(pil_image, hough_threshold=100):
    """
    Przyjmuje obraz PIL, wykrywa na nim linie i zwraca macierz rotacji (2x3 z OpenCV),
    która wyrównuje dominujące linie do pionu/poziomu.
    
    :param pil_image: Obiekt PIL.Image
    :param hough_threshold: Minimalna liczba głosów dla Transformaty Hougha 
                            (zmniejsz, jeśli nie wykrywa linii na prostych obrazach)
    :return: Tablica NumPy reprezentująca macierz przekształcenia afinicznego (2x3)
    """
    # 1. Konwersja obrazu PIL na format zgodny z OpenCV (NumPy array)
    img_np = np.array(pil_image)
    
    # 2. Konwersja do skali szarości
    if len(img_np.shape) == 3:
        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    elif len(img_np.shape) == 4:
        # Obsługa obrazów z kanałem alfa (np. PNG)
        gray = cv2.cvtColor(img_np, cv2.COLOR_RGBA2GRAY)
    else:
        gray = img_np
        
    h, w = gray.shape
    center = (w // 2, h // 2)

    # 3. Wykrywanie krawędzi (detektor Canny'ego)
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    plt.imshow(edges)
    plt.show()
    # 4. Transformata Hougha do wykrycia prostych linii
    lines = cv2.HoughLines(edges, 1, np.pi / 180, hough_threshold)
    # plt.imshow(lines)
    # plt.show()

    # Jeśli nie wykryto żadnych linii, zwracamy macierz braku rotacji
    if lines is None:
        print('jd')
        return cv2.getRotationMatrix2D(center, 0, 1.0)

    offsets = []
    for line in lines:
        rho, theta = line[0]
        angle_deg = np.degrees(theta)
        
        # Obliczamy odchylenie linii od najbliższego kąta prostego (0, 90, 180...)
        offset = angle_deg % 90
        
        # Dopasowanie zakresu odchylenia do [-45, 45] stopni
        if offset > 45:
            offset -= 90
            
        offsets.append(offset)

    # 5. Używamy mediany odchyleń, aby uodpornić się na pojedyncze linie pod dziwnymi kątami
    median_offset = np.median(offsets)
    
    # OpenCV wykonuje obrót w lewo dla wartości dodatnich. 
    # Aby skorygować obraz, musimy obrócić go w przeciwną stronę do wykrytego pochylenia.
    rotation_angle = -median_offset

    # 6. Wygenerowanie macierzy transformacji afinicznej dla obrotu
    rotation_matrix = cv2.getRotationMatrix2D(center, rotation_angle, 1.0)

    return rotation_matrix

In [ ]:
M = get_straightening_rotation_matrix(image)
print(M)
# Przekształć oryginalny obraz za pomocą OpenCV
img_cv = np.array(image)
h, w = img_cv.shape[:2]
straightened_img_cv = cv2.warpAffine(img_cv, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

# Zamień z powrotem na obiekt PIL i wyświetl
straightened_image = Image.fromarray(straightened_img_cv)
straightened_image.show()

In [ ]:
img = np.array(image)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# 1. Binaryzacja
thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                cv2.THRESH_BINARY_INV, 15, 5)

# 2. Szukanie linii poziomy i pionowych
kernel_length = img.shape[1] // 40
vert_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, kernel_length))
hori_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_length, 1))

img_vw = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, vert_kernel, iterations=2)
img_hw = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, hori_kernel, iterations=2)

In [ ]:
plt.imshow(img_vw)
plt.show()
plt.imshow(img_hw)
plt.show()